### Handin 2


# Info
Everything should be completed and approved in person. Groups are fine, as 1 random person will have to present on behalf of the group.

The objectives for this handin is:
* Investigate loss curves
* Linear Regression
* Feature Encoding 
* Simple Interface with Dash
* Speeding up with Numba


# Task 1

Prove that there exist an $\alpha \in R$ such that $y$ becomes 2.  (Taken from a math exam at BI Nydalen)

1) $\alpha x + y = 4$   
2) $-x + 3y = 2$  


### Answer Task 1

We impose $y = 2$ and check which $x$ and $\alpha$ follow.

**Step 1: find $x$ (equation 2, no $\alpha$):**
$-x + 3 \cdot 2 = 2 \implies -x = -4 \implies x = 4$

**Step 2: find $\alpha$ (equation 1):**
$\alpha \cdot 4 + 2 = 4 \implies 4\alpha = 2 \implies \alpha = \frac{1}{2}$

**Step 3: verification** with $\alpha = \frac{1}{2}$, $x = 4$, $y = 2$:
1) $\frac{1}{2} \cdot 4 + 2 = 4$ ✓  
2) $-4 + 3 \cdot 2 = 2$ ✓

**Uniqueness:** $\det \begin{pmatrix} \alpha & 1 \\ -1 & 3 \end{pmatrix} = 3\alpha + 1 = 2.5 \neq 0$,
so the system has exactly one solution, and in it $y = 2$.

Hence $\alpha = \frac{1}{2}$ exists such that $y = 2$. $\blacksquare$

# Task 2 -- Investigating the loss curve


We are going to investigate how an algorithm navigates the L2 loss curve.

To this end we will first use our very simple model $f_\theta(x) = \theta$ to model the training data given below.


### Task 2a
Visualize the model $f_\theta(x)$ with  $\theta=0.34$  alongside the training data in the plot below.


In [4]:
import numpy as np
import plotly.express as px


x_train = np.arange(0.0, 1.0, 0.025)
# numpy arr of shape (40,) 
y_train = 0.4 + x_train * 0.55 + np.random.randn(x_train.shape[0])*0.2


fig = px.scatter(x=x_train, y=y_train, title="train dataset")

# add to plot here
# const based on the fn
theta = 0.34

# crée un arr de la même taille que x_train et rempli avec la valeur de theta
y_pred = np.full_like(x_train, theta)

fig.add_scatter(x=x_train, y=y_pred, mode="lines", name="model θ=0.34")

fig.show()


### Task 2b

Create a plot that shows the loss curve for $\theta$ in the range [0, 1], using the Mean Squared Error loss function.  
That is, $L(x, y) = \frac{1}{m} \sum [ (f_\theta(x_k) - y_k)^2 ]$. Where $m$ is the number of data points in the training set. Remember: $f_{\theta}(x) = \theta$.


Using the plot find the value of $\theta$ that minimize the loss.

In [5]:
# -- CODE -- for Task 2b goes here.

# le modèle : prédit toujours theta, peu importe x (ligne horizontale)
def f(theta, x):
    return theta

# broadcasting : numpy "étire" le nombre pour le soustraire de chaque case
#   [0.75, 0.49, 0.62] - 0.5  ->  [0.25, -0.01, 0.12]
def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

# comme arange(0, 1, 0.01) mais avec 101 points et inclusif
thetas = np.linspace(0, 1, 101)

losses = []
for t in thetas:
    y_pred = f(t, x_train)        
    loss = mse(y_train, y_pred)   # 40 valeurs vs 1 nombre -> broadcasting -> 1 loss
    losses.append(loss)           # à la fin : 101 losses, une par theta


#show the loss curve
fig = px.line(x=thetas, y=losses, title="MSE loss as a function of θ",
              labels={"x": "θ", "y": "MSE"})
fig.show()

best = np.argmin(losses)
print("best θ =", thetas[best], "  loss =", losses[best])

best θ = 0.62   loss = 0.10368315981655485


### Task 2c 
We have the following loss curves (same loss function as in 2b):  

![title](loss_curves_mse.png)



# Model:
The model is of the form $f_\theta(x) = ax + b$ with $a,b \in \theta$. 
Here the different curves is the loss for:
1) Set $b = 0.1$ and $a \in [-1, 1]$.
2) Set $b = 0.75$ and $a \in [-1, 1]$.
3) Set $b = 1.5$ and $a \in [-1, 1]$.  
While a is between [-1, 1] (the x-axis in the plot).


Objective: Find a set of datapoints that duplicate these graphs


In [ ]:
# -- CODE for 2c ---
import plotly.graph_objects as go
import numpy as np

# notation : y = m*x + h   (dans l'énoncé : m = a, h = b)
def f_line(m, x, h):
    return m * x + h

# les points : sur la droite y = 0.5x + 0.75 (indice courbe orange),
# le 0.6 est choisi car il est proche de la droite orange, mais pas exactement dessus
# centrés autour de x ≈ 0.6 (indice courbe verte), écartés pour que la verte ne touche pas 0
x_data = np.array([0.4, 0.8])
y_data = f_line(0.5, x_data, 0.75)      # -> [0.95, 1.15]

m_values = np.linspace(-1, 1, 101)       # comme le prof : m (= a) de -1 à 1

fig = go.Figure()
for h in [0.1, 0.75, 1.5]:               # une courbe par valeur de h (= b)
    losses = []
    for m in m_values:
        y_pred = f_line(m, x_data, h)    # ce que prédit la ligne (m, h) pour chaque x_data
        losses.append(mse(y_data, y_pred))
    fig.add_scatter(x=m_values, y=losses, mode="lines", name=f"h={h}")

fig.update_layout(title="My loss curves", xaxis_title="m - parameter (a)", yaxis_title="L2 loss")
fig.show()


**Answer 2c:** The datapoints $(0.4, 0.95)$ and $(0.8, 1.15)$ reproduce the curves.

- **Orange curve ($b = 0.75$)** reaches ≈ 0 at $a = 0.5$, so the points lie on $y = 0.5x + 0.75$.
- **Green curve ($b = 1.5$)** is minimal at $a ≈ -0.7$: the best line $y = 1.5 - 0.7x$ crosses
  $y = 0.5x + 0.75$ at $x ≈ 0.625$, so the points are centred around $x ≈ 0.6$.
- The green minimum is not 0, so there is more than one point: they must be spread out.

# Task 3
Train a  linear regression with a L2 loss on the training data using Gradient Descent. 
The code below should give a (non-vectorized) on how it is found.  

The gradient is found as:  
$ L = \frac{1}{2}(\hat{y} - y )^2 $  

$ \hat{y} = f_\theta(x) = \theta$  


$ \frac{\partial L}{\partial\theta} = \frac{\partial L}{\partial \hat{y}} \frac{\partial \hat{y}}{\partial \theta}$ (using the chain rule).

With:  
$\frac{\partial L}{\partial \hat{y}} = (\hat{y} - y) \times 1 = (\hat{y} - y)$  

$\frac{\partial \hat{y}}{\partial \theta} = 1  $


Gives us:  
$\frac{\partial L}{\partial\theta} = (\hat{y} - y)$


### Questions:
1) Draw the graph/tree that shows how these partial derivatives are connected. 
2) Find a set of hyperparameters that converge for $\theta_{\text{init}} = 5.5$.  How can we determine if the algorithm has converged?  
3) Can you find a learning rate that the algorithm does not converge for?  
4) What is the "best" learning rate for this particular dataset?  
5) You might be asked to show how the gradients flow in another loss/function : be prepared.



In [ ]:
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


def gradient_of_J(theta, x, y):
    # 
    y_hat = theta
    
    # dL / dy_hat
    dLdy = (y_hat - y)
    
    # dy_hat / dTheta
    dy_HatdTheta = 1
    
    # chain rule
    dLdTheta = dLdy * dy_HatdTheta
    
    return dLdTheta


def calculate_l2_loss_non_vectorized(theta, xs, ys):
    loss = 0.0
    for k in range(ys.shape[0]):
        y_pred = theta 
        loss += (y_pred - ys[k])**2

    
    mean_loss = loss / ys.shape[0]
    return mean_loss


    

initial_theta = 5.5

learning_rate = 0.1
theta = np.array([initial_theta])
m = x_train.shape[0]
n_steps = 1000

print("Running GD with initial theta: {:.2f}, learning rate: {} over {} datapoints for {} steps".format(
    theta.item(),
    learning_rate,
    m,
    n_steps))



search_history = []
for steps in range(n_steps):    
        
    gradient_theta_sum = 0.0
    for k in range(m):
        gradient_theta_sum += gradient_of_J(theta, x_train[k], y_train[k]) 

    mean_gradient = (1/m) * gradient_theta_sum
    
    # compute the step size
    step_size = abs(learning_rate * mean_gradient.item())
    if step_size < 0.00001:
        print("convergé après", steps, "pas, θ =", theta.item())
        break


    loss = calculate_l2_loss_non_vectorized(theta, x_train, y_train)

    print("[visit] theta: {:.2f} => loss: {:.2f}".format(theta.item(), loss.item()))

    # update theta using GD
    theta = theta - (learning_rate * mean_gradient)
    search_history.append((theta, loss))

    


# quick helper to generate plots 
loss_x = np.arange(-4, 6, 0.01)

loss_y = np.array([calculate_l2_loss_non_vectorized(t, x_train, y_train) for t in loss_x])

fig = px.line(x=loss_x, y=loss_y, title="GD History : Marks are iterations.")


x_visit, _ = list(zip(*search_history))
x_visit = np.concatenate(x_visit)
y_visit = np.array([calculate_l2_loss_non_vectorized(t, x_train, y_train) for t in x_visit])

fig.add_trace(go.Scatter(x=x_visit, y=y_visit, name='GD history',
                         line = dict(color='firebrick', width=8, dash='dot')))

fig.show()

Running GD with initial theta: 5.50, learning rate: 0.1 over 40 datapoints for 10 steps
[visit] theta: 5.50 => loss: 23.10
[visit] theta: 5.02 => loss: 18.72
[visit] theta: 4.59 => loss: 15.17
[visit] theta: 4.20 => loss: 12.30
[visit] theta: 3.85 => loss: 9.97
[visit] theta: 3.53 => loss: 8.08
[visit] theta: 3.25 => loss: 6.55
[visit] theta: 2.99 => loss: 5.32
[visit] theta: 2.77 => loss: 4.31
[visit] theta: 2.56 => loss: 3.50


In [ ]:
# -- Q3 / Q4 : on compare plusieurs learning rates, toujours en partant de θ = 5.5 --

def run_gd(learning_rate, theta=5.5, max_steps=1000, tol=1e-5):
    for step in range(max_steps):
        # même gradient que plus haut (θ - y), calculé sur les 40 points d'un coup puis moyenné
        mean_gradient = np.mean(gradient_of_J(theta, x_train, y_train))
        if abs(learning_rate * mean_gradient) < tol:   # le pas est minuscule : convergé
            return theta, step
        theta = theta - learning_rate * mean_gradient
        if abs(theta) > 1e6:                           # θ explose : divergé
            return theta, None
    return theta, None                                 # toujours pas convergé après max_steps

print("moyenne de y_train =", round(np.mean(y_train), 4))
for lr in [0.01, 0.1, 0.5, 1.0, 1.5, 1.9, 2.0, 2.1]:
    theta_final, steps = run_gd(lr)
    if steps is None:
        print(f"lr = {lr:<4} -> NE converge PAS   (θ = {theta_final:.4g})")
    else:
        print(f"lr = {lr:<4} -> converge en {steps:>3} pas, θ = {theta_final:.4f}")

### Answer Task 3

**Q1: Graph of the partial derivatives**

```text
            ∂ŷ/∂θ = 1                 ∂L/∂ŷ = ŷ − y
   θ  ─────────────────►  ŷ = θ  ─────────────────►  L = ½(ŷ − y)²
                                                         ▲
                                                         │
                                                         y   (data, fixed)

Chain rule (backwards):  ∂L/∂θ = ∂L/∂ŷ × ∂ŷ/∂θ = (ŷ − y) × 1 = ŷ − y = θ − y
```

Forward (left → right) we compute the prediction and the loss.
Backward (right → left) we multiply the local derivatives along the path to get the gradient (chain rule).

**Q2: Hyperparameters that converge from $\theta_{init} = 5.5$**

- `learning_rate = 0.1`, `n_steps = 1000` (used as a maximum, the loop stops earlier).
- With the original 10 steps it does **not** converge ($\theta \approx 2.56$): with `learning_rate = 0.1`, each step
  only removes 10 % of the remaining distance to the minimum ($0.9^{10} \approx 0.35$ of it is still left).
- **How to know it has converged:** stop when the step $|\text{learning\_rate} \times \text{gradient}|$ is smaller
  than $10^{-5}$, i.e. $\theta$ does not move anymore (the gradient is ≈ 0, the loss stops decreasing).
- Result: it converges after about **103 steps**, and $\theta$ ends at the **mean of `y_train`**,
  the same value as the minimum found with the plot in Task 2b.

**Q3: A learning rate that does not converge** (see the comparison cell above)

- Any `learning_rate ≥ 2`. At exactly 2, $\theta$ bounces forever between two values (5.5 and about −4);
  above 2 (e.g. 2.1) every step jumps further than the previous one and $\theta$ explodes.
- Why: the update gives $\theta_{new} - \bar{y} = (1 - \text{lr})(\theta - \bar{y})$, so the distance to the minimum
  $\bar{y}$ is multiplied by $(1 - \text{lr})$ at every step. It only shrinks if $|1 - \text{lr}| < 1$, i.e. $0 < \text{lr} < 2$.
- Between 1 and 2 the factor is negative: $\theta$ zigzags around the minimum but still converges.

**Q4: The "best" learning rate for this dataset**

- `learning_rate = 1` converges in **one single step**: $\theta - 1 \cdot (\theta - \bar{y}) = \bar{y}$, the minimum directly.
- This only works because the model $f_\theta(x) = \theta$ is so simple. For real models (Task 4, KKD)
  the best learning rate cannot be computed in advance: we try several values and keep the one that converges fastest
  without diverging.

**Q5: Gradients for another model / loss**

The recipe is always the same: **∂L/∂(parameter) = ∂L/∂ŷ × ∂ŷ/∂(parameter)**.

*Other model: the line* $\hat{y} = a x + b$ (two parameters), with the same loss $L = \frac{1}{2}(\hat{y} - y)^2$:

```text
   a ──(∂ŷ/∂a = x)──┐
                    ├──►  ŷ = a·x + b  ──(∂L/∂ŷ = ŷ − y)──►  L = ½(ŷ − y)²  ◄── y
   b ──(∂ŷ/∂b = 1)──┘

∂L/∂a = (ŷ − y) × x          ∂L/∂b = (ŷ − y) × 1
```

Update (averaged over the $m$ points): $a \leftarrow a - \text{lr} \cdot \frac{1}{m}\sum (\hat{y}_k - y_k)\,x_k$ and
$b \leftarrow b - \text{lr} \cdot \frac{1}{m}\sum (\hat{y}_k - y_k)$.

*Other loss: L1* $L = |\hat{y} - y|$ with $\hat{y} = \theta$: only the first factor changes,
$\frac{\partial L}{\partial \hat{y}} = \text{sign}(\hat{y} - y)$ (+1 or −1), so $\frac{\partial L}{\partial \theta} = \text{sign}(\theta - y) \times 1$.

The error term $(\hat{y} - y)$ always comes first; it is multiplied by what the parameter "touches" ($x$ for the slope, 1 for the bias).

## Task 4: Gradient Descent
Below is a simple vectorized impl. of GD that *CAN* be used as a starting point. 
Please make sure you understand exactly HOW it works (so that you could have implemented one yourself).
(Note that it uses a augmented matrix to skip the bias term).

1) Change the code to handle the bias parameter directly. (No agumented matrix).
2) Change the code to use Stochastic Gradient Descent with mini-batches. (Batch size should be 2 or more).
3) Re-organize the code and add numba as to make the SGD go pew pew (faster). To make it easier for yourself - numba does not play nice with Jupyter, so consider running it in its own python file.
4) *OPTIONAL* numba also supports GPU (https://numba.readthedocs.io/en/stable/cuda) implement GPU acceleration for SDG. (again: numba dont like notebooks)

Numba: https://numba.readthedocs.io/en/stable/


In [ ]:
import numpy as np
import plotly.express as px

# prédiction pour TOUS les points d'un coup : chaque ligne de xs fois theta
# avec xs = [1, x] et theta = [b, a]  ->  ŷ = b·1 + a·x
def predict(theta, xs): 
    return np.dot(xs, theta)

# la loss L2 : SOMME des erreurs au carré (pas encore la moyenne)
def J_squared_residual(theta, xs, y):
    h = predict(theta, xs)         # h = ŷ, les prédictions
    sr = ((h - y)**2).sum()    
    return sr

# les gradients de b ET de a en une seule ligne (chain rule de la Q5, Task 3) :
#   colonne des 1 -> Σ (ŷ − y)·1 = gradient de b
#   colonne des x -> Σ (ŷ − y)·x = gradient de a
def gradient_J_squared_residual(theta, xs, y):
    h = predict(theta, xs) 
    grad = np.dot(xs.transpose(), (h - y)) 
    return grad


# the dataset (already augmented so that we get a intercept coef)
# remember: augmented x -> we add a colum of 1's instead of using a bias term.
# 3 points (x = 0.5, 1, 2) ; 1re colonne = les 1 du biais/intercept (sert à rien), 2e colonne = x
# les y suivent exactement y = 1·x + 0.5  ->  on doit trouver b = 0.5, a = 1
# y = mx + h => y = a·x + b => 1.0 = a·x + b  ->  b = 0.5, a = 1 (tjr 1)
data_x = np.array([[1.0, 0.5], [1.0, 1.0], [1.0, 2.0]])
data_y = np.array([[1.0], [1.5], [2.5]])
n_features = data_x.shape[1]       # 2 colonnes -> 2 paramètres (b et a)

# variables we need 
theta = np.zeros((n_features, 1))  # theta = [b, a], on part de [0, 0]
learning_rate = 0.1
m = data_x.shape[0]                # nombre de points (3)

# run GD
j_history = []                     # la loss à chaque itération (pour le graphique)
n_iters = 10
for it in range(n_iters):
    j = J_squared_residual(theta, data_x, data_y)
    j_history.append(j)
    
    # mise à jour : theta ← theta − lr × gradient MOYEN (le 1/m transforme la somme en moyenne)
    theta = theta - (learning_rate * (1/m) * gradient_J_squared_residual(theta, data_x, data_y))
    
print("theta shape:", theta.shape)

# append the final result.
j = J_squared_residual(theta, data_x, data_y)
j_history.append(j)
print("The L2 error is: {:.2f}".format(j))


# find the L1 error.
# L1 = somme des |erreurs| (valeur absolue au lieu du carré)
y_pred = predict(theta, data_x)
l1_error = np.abs(y_pred - data_y).sum()
print("The L1 error is: {:.2f}".format(l1_error))


# Find the R^2 
# if the data is normalized: use the normalized data not the original data (task 3 hint).
# https://en.wikipedia.org/wiki/Coefficient_of_determination
# R² = 1 − (erreur du modèle) / (erreur si on prédisait toujours la moyenne)
#   1 = parfait, 0 = pas mieux que la moyenne
u = ((data_y - y_pred)**2).sum()             # erreur du modèle
v = ((data_y - data_y.mean())** 2).sum()     # erreur du « modèle moyenne »
print("R^2: {:.2f}".format(1 - (u/v)))


# plot the result
# courbe de la loss au fil des itérations : elle doit descendre
fig = px.line(j_history, title="J(theta) - Loss History")
fig.show()


theta shape: (2, 1)
The L2 error is: 0.03
The L1 error is: 0.25
R^2: 0.97


In [ ]:
# -- Task 4.1 : le biais h est une variable à part (plus de colonne de 1) --
import numpy as np
import plotly.express as px

# le modèle : y = m·x + h
def predict(m, h, xs):
    return np.dot(xs, m) + h

# la loss L2 : somme des erreurs au carré
def J_squared_residual(m, h, xs, y):
    pred = predict(m, h, xs)
    return ((pred - y) ** 2).sum()

# les gradients (chain rule, Q5 de la Task 3)
def gradients(m, h, xs, y):
    pred = predict(m, h, xs)
    grad_m = np.dot(xs.T, (pred - y))   # Σ (ŷ − y) × x
    grad_h = (pred - y).sum()           # Σ (ŷ − y) × 1
    return grad_m, grad_h

# les données : seulement les vrais x (plus de colonne de 1)
data_x = np.array([[0.5], [1.0], [2.0]])
data_y = np.array([[1.0], [1.5], [2.5]])

m = np.zeros((1, 1))   # la pente, départ à 0
h = 0.0                # le biais, départ à 0
learning_rate = 0.1
n = data_x.shape[0]    # 3 points

j_history = []
for it in range(10):
    j_history.append(J_squared_residual(m, h, data_x, data_y))
    grad_m, grad_h = gradients(m, h, data_x, data_y)
    m = m - learning_rate * (1/n) * grad_m     # on corrige la pente
    h = h - learning_rate * (1/n) * grad_h     # on corrige le biais

print("m =", m.item(), "  h =", h)
print("L2 error:", J_squared_residual(m, h, data_x, data_y))

fig = px.line(j_history, title="4.1 - Loss History (bias handled directly)")
fig.show()

**Answer 4.1 (bias handled directly):** the column of 1's is removed from `data_x`, and the bias `h` becomes its own
variable: $\hat{y} = m x + h$. Its gradient is simply the sum of the errors, because $\partial \hat{y} / \partial h = 1$:
$\frac{\partial J}{\partial h} = \sum (\hat{y} - y)$, while $\frac{\partial J}{\partial m} = \sum (\hat{y} - y)\,x$.
After 10 iterations we get exactly the same values as the augmented version ($m \approx 0.862$, $h \approx 0.597$),
which shows both versions are equivalent.

In [ ]:
# -- Task 4.2 : Stochastic Gradient Descent avec des mini-batches --
# on réutilise predict, J_squared_residual et gradients de la 4.1 (cellule au-dessus)

# plus de données pour que les mini-batches aient du sens : 200 points autour de y = 1·x + 0.5
rng = np.random.default_rng(0)                       # générateur aléatoire (seed 0 = mêmes résultats à chaque fois)
data_x = rng.uniform(0, 2, size=(200, 1))            # 200 x entre 0 et 2
data_y = 1.0 * data_x + 0.5 + rng.normal(0, 0.1, size=(200, 1))   # la droite + un peu de bruit

m = np.zeros((1, 1))   # la pente, départ à 0
h = 0.0                # le biais, départ à 0
learning_rate = 0.1
batch_size = 8         # nombre de points par paquet (2 ou plus)
n_epochs = 20          # 1 epoch = on a utilisé chaque point une fois
n = data_x.shape[0]    # 200 points

mse_history = []
for epoch in range(n_epochs):
    order = rng.permutation(n)                     # on mélange l'ordre des points (le "stochastic")
    for start in range(0, n, batch_size):          # on avance de 8 en 8 : 0, 8, 16, ...
        idx = order[start:start + batch_size]      # les numéros des 8 points du paquet
        xb, yb = data_x[idx], data_y[idx]          # le paquet (mini-batch)
        grad_m, grad_h = gradients(m, h, xb, yb)   # gradients calculés sur le paquet SEULEMENT
        m = m - learning_rate * (1/len(idx)) * grad_m
        h = h - learning_rate * (1/len(idx)) * grad_h
    # à la fin de chaque epoch : l'erreur moyenne (MSE) sur TOUS les points
    mse_history.append(J_squared_residual(m, h, data_x, data_y) / n)

print("m =", m.item(), "  h =", h, "  (cible : m = 1, h = 0.5)")
print("MSE finale :", mse_history[-1])

fig = px.line(mse_history, title="4.2 - Mini-batch SGD : MSE à la fin de chaque epoch")
fig.show()

**Answer 4.2 (mini-batch SGD):** instead of using all the points for every step, each step uses a small random
batch (`batch_size = 8`). At every **epoch** the points are shuffled (`rng.permutation`), split into batches of 8,
and we take one gradient step per batch, averaging the gradient over the batch (`1/len(idx)`).
With 200 points that is 25 steps per epoch. Each step is less precise (the gradient is estimated from only 8 points),
but much cheaper, which matters when there are many data points.
Data: 200 points around $y = 1 \cdot x + 0.5$ with noise. After 20 epochs: $m \approx 0.994$, $h \approx 0.508$
(target $m = 1$, $h = 0.5$); the MSE stops around 0.01, which is the noise level ($0.1^2$).

**Answer 4.3 (numba):** the code is in `task4_numba.py` (numba does not play well with Jupyter).
Run it from the `handin2` folder with `.venv/bin/python task4_numba.py`.

- The mini-batch SGD of 4.2 is re-organized into one function `sgd(x, y, learning_rate, batch_size, n_epochs)`
  written with simple loops over the points of each batch (loops are exactly what numba speeds up).
- `sgd_fast = njit(sgd)` compiles the **same** function to machine code. The first call compiles it,
  so it is not timed.
- On 100 000 points, 5 epochs, batch size 8: plain Python ≈ 0.20 s, numba ≈ 0.007 s → about **30× faster**,
  with the same result ($m \approx 1$, $h \approx 0.5$).

# Task 5

For a two layer network (as shown at: https://opentsetlin.com:5577/ ) with a ReLu activation function.  
Find a recipe for calculating the parameters the network needs to create a V shape with the lowest point at a spesifc location. 
Be ready to explain how/why your recipe works. 
.  
.  
.  




In [ ]:
# -- Task 5 : recette pour un V avec deux neurones ReLU, fond en x0 --
import numpy as np
import plotly.graph_objects as go

def relu(z):
    return np.maximum(0, z)              # négatif -> 0, positif -> on garde

# le réseau du site du prof : chaque ligne de W1 = un neurone [m, h], W2 = [u1, u2]
def network(x, W1, W2):
    a1 = relu(W1[0][0] * x + W1[0][1])   # neurone 1 : ReLU(m1·x + h1)
    a2 = relu(W1[1][0] * x + W1[1][1])   # neurone 2 : ReLU(m2·x + h2)
    return W2[0] * a1 + W2[1] * a2, a1, a2

# la recette
def v_recipe(x0, slope=1.0):
    W1 = [[1, -x0],                      # neurone 1 : rampe qui monte vers la droite, coude en x0
          [-1, x0]]                      # neurone 2 : rampe qui monte vers la gauche, coude en x0
    W2 = [slope, slope]                  # on additionne les deux rampes (slope = raideur du V)
    return W1, W2

x0 = 5
W1, W2 = v_recipe(x0)
x = np.linspace(0, 10, 201)              # même intervalle que le site du prof
y, a1, a2 = network(x, W1, W2)

print("W1 =", W1, "  W2 =", W2)
print("point le plus bas : x =", x[np.argmin(y)], "  y =", y.min())

fig = go.Figure()
fig.add_scatter(x=x, y=a1, mode="lines", name="neurone 1 : ReLU(x - x0)", line=dict(dash="dot"))
fig.add_scatter(x=x, y=a2, mode="lines", name="neurone 2 : ReLU(-x + x0)", line=dict(dash="dot"))
fig.add_scatter(x=x, y=y, mode="lines", name="sortie : le V", line=dict(width=4))
fig.update_layout(title=f"Task 5 : V avec le fond en x0 = {x0}", xaxis_title="x", yaxis_title="y")
fig.show()

### Answer Task 5 (recipe for a V shape)

**The network** (same as on the demo site): $y = u_1\,\text{ReLU}(m_1 x + h_1) + u_2\,\text{ReLU}(m_2 x + h_2)$.
On the site, row $k$ of $W_1$ is $[m_k, h_k]$ (because the input is $[x, 1]$) and $W_2 = [u_1, u_2]$.

**Recipe for a V with its lowest point at $x_0$:**

| | weight $m$ | bias $h$ | output weight $u$ |
|---|---|---|---|
| neuron 1 | $1$ | $-x_0$ | $1$ |
| neuron 2 | $-1$ | $x_0$ | $1$ |

Example $x_0 = 5$: $W_1 = [[1, -5], [-1, 5]]$, $W_2 = [1, 1]$. Checked on the site: for $x = 0, 1, \dots, 10$ the output is
$5, 4, 3, 2, 1, 0, 1, 2, 3, 4, 5$.

**Why it works:**

1. One ReLU neuron is a **ramp**: flat at 0 on one side, a straight line on the other. The kink is where
   $m x + h = 0$, i.e. at $x = -h/m$.
2. A V has **two arms**, so we need **two neurons**, one per arm. The sign of $m$ chooses the direction:
   $m = +1$ rises to the right, $m = -1$ rises to the left.
3. $h = -m\,x_0$ puts **both kinks at $x_0$**.
4. Left of $x_0$ only neuron 2 is on (the output goes down), right of $x_0$ only neuron 1 is on (the output goes up),
   and at $x_0$ both are 0: that is the lowest point. In fact $\text{ReLU}(x - x_0) + \text{ReLU}(x_0 - x) = |x - x_0|$.
5. $u_1 = u_2 = s > 0$ sets the steepness ($s\,|x - x_0|$); different values give a tilted V, negative values flip it upside down.

**Limitation:** there is no bias on the output ($y = u_1 a_1 + u_2 a_2$), so at $x_0$ the output is always $0$.
The V can be moved left and right, but not up or down; that would need an output bias.

# KKD Real Estate

### Note: no pandas, sklearn or similar libraries should be used, numpy, dash, numba and plotly should be sufficient. Ask if you wonder about a library.
### Implementing your own SGD/GD is a core component of this task.

The project consists of 5 parts: 

1) 
Go though the data and understand how encode the various features. 
* Clean the data for potential noise and simply wrong input.
* Make sure you identify how a linear classifier will be affected by the encoding scheme. 
* How do you handle missing data?
* How are the different features connected?
* Encode the features.

2) 
Identify objectives that could be valuable for KKD Real Estate, the objectives comes in two flavors:
* Answering questions, such as: What is a fair price for our ad packages? How is our best agent?
* Creating dashboards/interfaces that for instance: can predict the marked price for a house or maybe tell 
the probability of the house being sold in X amount of days?
* To pass the handin you at least need to implement a price model. 

3) 
Train one/many linear model(s) based on the data to solve the objectives.

4) 
Implement the dashboard interface such that we can input the nessesary parameters.
(See 'kkd_dashboard.py' on canvas for a potential starting point for the dashboard.)  
This interface should be powered by your models.

5) 
A client wants to know how the price model works, my inspecting the weights (that is: $\theta$) give a 
overview of the most important factors in the model. What are *key-drives* in the model?


### Deliverables
A presentation of the findings, a short written log of the work (bullet points / jupyter), a dashboard, and all code that you have.


### Dataset
* agents.jsonl - the real estate agents.
* districts.jsonl - the city districts
* houses.jsonl - the houses that has been on the marked the last year
* schools.jsonl - info about the schools in the districts


